# Drive failure from 30-day SMART windows — Backblaze, 2022 to today

The same pipeline as `Dataset_backblaze_Analysis.ipynb`, restricted to archives from
**2022 onward** so a full run is a fraction of the cost, plus four things that notebook
does not do:

| § | question | what gets printed |
|---|---|---|
| **1** | is any column a proxy for `failure`? | the leakage audit and the surviving feature set $X$ |
| **2** | can the model see the label at all? | **four executed checks**, not a claim |
| **3** | how many 30-day windows cover a drive's whole life? | per-drive window and **batch counts** |
| **4** | does this drive fail in *this* 30 days? | a verdict per window, plus per-drive histograms |
| **5** | what does a tree grown to 100% accuracy look like? | the depth ladder, and what it costs |

**Scope.** Archives `Q1_2022` through the newest quarter. A drive that was already in
service in 2021 enters with only its 2022-onward rows, which is the intended reading of
"from 2022 till today".

**Still the fleet, not a cohort.** No vendor filter and no matched sampling: failures are
a fraction of a percent of drive-days, as they are in a real datacentre.

In [1]:
import gc, json, os, sys, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "scripts")
import leakage_audit as la
import backblaze_window_pipeline as bw
import archive_window_pipeline as ap

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 9})
pd.set_option("display.width", 220, "display.max_columns", 40)

SHARD_DIR  = os.environ.get("BACKBLAZE_SHARDS",
                            os.path.join("data", "backblaze_archive", "shards"))
STORE_DIR  = os.environ.get("BACKBLAZE_STORE_2022",
                            os.path.join("data", "backblaze_archive", "store_2022"))
SINCE_YEAR = int(os.environ.get("SINCE_YEAR", "2022"))
FIG_DIR    = "results/figures"
SEED       = 42
os.makedirs(FIG_DIR, exist_ok=True)
bw.set_seed(SEED)

shards = ap.shard_paths(SHARD_DIR, since_year=SINCE_YEAR)
print(f"torch {torch.__version__} | device "
      f"{'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"{len(shards)} shards from {SINCE_YEAR} on: "
      f"{os.path.basename(shards[0])[:-8]} -> {os.path.basename(shards[-1])[:-8]}")
print("  " + ", ".join(os.path.basename(p)[:-8] for p in shards))

torch 2.14.0.dev20260708+cu132 | device cuda
18 shards from 2022 on: Q1_2022 -> Q2_2026
  Q1_2022, Q2_2022, Q3_2022, Q4_2022, Q1_2023, Q2_2023, Q3_2023, Q4_2023, Q1_2024, Q2_2024, Q3_2024, Q4_2024, Q1_2025, Q2_2025, Q3_2025, Q4_2025, Q1_2026, Q2_2026


---
## 1 — Target-leakage audit

A leaked column is not *evidence about* the outcome but a *restatement of* it. In drive
telemetry it is almost never a SMART attribute; it hides in three places, and the audit
probes all three: the column values, the **missingness** pattern, and **record
geometry** — anything derived from where a row sits relative to the end of a drive's
record, because Backblaze writes `failure = 1` on a drive's last reporting day.

The audit samples by **drive**, never by drive-day: every geometry probe is derived from
a drive's own first and last day, and a drive whose rows have been thinned has neither.

In [2]:
t0 = time.time()
cfg0 = ap.ArchiveConfig(shard_dir=SHARD_DIR, store_dir=STORE_DIR,
                        since_year=SINCE_YEAR, random_state=SEED)
census = ap.build_drive_census(cfg0)

epoch = pd.Timestamp("1970-01-01")
print(f"\n{len(census):,} drives | {int(census.n_obs.sum()):,} observed drive-days | "
      f"{int(census.has_failure.sum()):,} drives with a failure event")
print(f"{epoch + pd.Timedelta(days=int(census.start_day.min())):%Y-%m-%d} -> "
      f"{epoch + pd.Timedelta(days=int(census.end_day.max())):%Y-%m-%d}"
      f"   ({time.time()-t0:.0f}s)")

  [ 1/18] Q1_2022.parquet    18,845,260 rows ->  212,404 drives  (  2.8s)


  [ 2/18] Q2_2022.parquet    19,424,436 rows ->  221,509 drives  (  2.8s)


  [ 3/18] Q3_2022.parquet    20,591,757 rows ->  232,016 drives  (  3.0s)


  [ 4/18] Q4_2022.parquet    21,496,309 rows ->  236,495 drives  (  3.4s)


  [ 5/18] Q1_2023.parquet    21,454,992 rows ->  242,703 drives  (  3.1s)


...

  [15/18] Q3_2025.parquet    29,844,451 rows ->  332,915 drives  (  4.8s)


  [16/18] Q4_2025.parquet    30,941,708 rows ->  341,664 drives  (  5.2s)


  [17/18] Q1_2026.parquet    30,597,484 rows ->  351,095 drives  (  4.7s)


  [18/18] Q2_2026.parquet    31,968,003 rows ->  359,101 drives  (  6.6s)


  census: 416,428 drives, 19,024 with a failure event

416,428 drives | 457,380,330 observed drive-days | 19,024 drives with a failure event
2022-01-01 -> 2026-06-30   (76s)


In [3]:
audit_df = ap.sample_audit_frame(cfg0, census, n_drives=15_000)
report = la.audit_frame(audit_df)
report.print_report()

  [ 1/18] Q1_2022.parquet    669,411 rows kept


  [ 2/18] Q2_2022.parquet    691,592 rows kept


  [ 3/18] Q3_2022.parquet    735,216 rows kept


  [ 4/18] Q4_2022.parquet    768,652 rows kept


  [ 5/18] Q1_2023.parquet    767,257 rows kept


...

  [16/18] Q4_2025.parquet  1,116,665 rows kept


  [17/18] Q1_2026.parquet  1,104,216 rows kept


  [18/18] Q2_2026.parquet  1,156,039 rows kept


  audit frame: 16,417,355 drive-days over 15,000 drives, 656 failure events (base rate 0.00400%)
[audit] scoring 23 telemetry columns, 6 identifier/topology columns, and 6 derived record-geometry probes over 16,417,355 drive-days ...


STEP 1 -- TARGET LEAKAGE AND PROXY AUDIT
  drive-days ........... 16,417,355
  distinct drives ...... 15,000
  failure events ....... 656 (base rate 0.0040%)

  thresholds: severe if rank-AUC >= 0.95, or 1-threshold balanced acc >= 0.90,
              or |r| >= 0.50, or missingness gap >= 0.25, or level lift >= 20x

--------------------------------------------------------------------------------------------
A. TELEMETRY COLUMNS vs failure -- top 12 by rank AUC
--------------------------------------------------------------------------------------------
       column coverage  n_unique   corr auc_directional balanced_acc verdict
smart_187_raw   0.3930      1120 0.0174          0.8887       0.8785 suspect
smart_183_raw   0.0325       169 0.0006          0.7950       0.7586   clean
smart_197_raw   0.9685      1396 0.0651          0.7862       0.7842   clean
  smart_5_raw   0.9910      6815 0.0212          0.7859       0.7807   clean
smart_198_raw   0.9902       890 0.0640          0.7004  

In [4]:
features = la.select_feature_columns(audit_df, report)

print(f"{len(features)} attributes survive coverage >= {report.cfg.min_coverage:.0%} "
      f"and the blocklist:\n")
tbl = report.columns.set_index("column")
for c in features:
    r = tbl.loc[c]
    print(f"  {c:<18} coverage {100*r.coverage:6.2f}%  "
          f"{int(r.n_unique):>9,} distinct  rank AUC {r.auc_directional:.4f}")

report.assert_clean(features)          # raises if a blocked name reached X
la.plot_audit(report, save_path=f"{FIG_DIR}/2022_step1_leakage_audit.png")
del audit_df; gc.collect()

15 attributes survive coverage >= 50% and the blocklist:

  smart_197_raw      coverage  96.85%      1,396 distinct  rank AUC 0.7862
  smart_5_raw        coverage  99.10%      6,815 distinct  rank AUC 0.7859
  smart_198_raw      coverage  99.02%        890 distinct  rank AUC 0.7004
  smart_4_raw        coverage  98.77%        423 distinct  rank AUC 0.6590
  smart_12_raw       coverage  99.82%        346 distinct  rank AUC 0.6403
  smart_9_raw        coverage  99.82%     82,565 distinct  rank AUC 0.6211
  smart_240_raw      coverage  65.79%     87,644 distinct  rank AUC 0.6022
  smart_193_raw      coverage  98.67%    136,319 distinct  rank AUC 0.5959
  smart_1_raw        coverage  99.74%  5,427,754 distinct  rank AUC 0.5781
  smart_7_raw        coverage  98.77%  5,873,176 distinct  rank AUC 0.5710
  smart_3_raw        coverage  98.77%      3,417 distinct  rank AUC 0.5443
  smart_192_raw      coverage  99.49%     16,010 distinct  rank AUC 0.5293
  smart_194_raw      coverage  99.82%     

  saved results/figures/2022_step1_leakage_audit.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\leakage_audit.py:618: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


28

---
## 2 — Building the store, then **proving** the label cannot be read

The dense matrix is built once and memory-mapped. `failure` is written to a *separate
file* from the feature matrix — it is not a column of `X` that gets dropped later, it
never enters `X` at all.

That is a claim, so the next cell tests it four ways:

1. **Names.** No channel is called `failure` or derived from it.
2. **Storage.** `values.f32` and `failure.u8` are different files and share no memory, so
   no write to one can surface in the other.
3. **Independence.** The label array is replaced with random noise and the same windows
   are gathered again. If a single byte of `X` depended on the label, `X` would move. It
   must come back bit-identical.
4. **Separability.** Every channel is scored against the window label on its own. A
   column that *was* the label would sit at AUC ≈ 1.0. The check fails the notebook if
   any channel exceeds 0.95.

Checks 1, 2 and 4 raise on failure; check 3 is the one that actually settles it, because
it is an experiment on the running code rather than an inspection of it.

In [5]:
cfg = ap.ArchiveConfig(
    shard_dir         = SHARD_DIR,
    store_dir         = STORE_DIR,
    since_year        = SINCE_YEAR,
    window_days       = 30,     # a sample is 30 consecutive days
    horizon_days      = 30,     # positive if the drive fails within 30 days of window end
    stride_days       = 1,      # every possible start is enumerated for eval
    min_days          = 30,     # shorter drives are dropped, not padded
    delta_lags        = (1, 7),
    add_observed_mask = True,
    feature_columns   = tuple(features),
    log1p_columns     = tuple(c for c in features
                              if c not in ("smart_194_raw", "smart_190_raw")),
    n_buckets         = 48,
    samples_per_drive = 4,
    train_positive_ratio = 0.25,
    batch_size        = 4096,
    random_state      = SEED,
)

store = ap.build_archive_store(cfg)

STEP A -- DRIVE CENSUS
  census cached: 416,428 drives

STEP B -- DENSE LAYOUT


  413,045 drives kept, 3,383 dropped (shorter than 30 days)
  459,048,144 dense drive-days (457,317,880 observed, 1,730,264 forward-filled)
  matrix 459,048,144 x 46 float32 = 84.5 GB memory-mapped

STEP C -- EXTERNAL SORT INTO 48 BUCKETS


  [ 1/18] Q1_2022.parquet    18,842,400 rows bucketed ( 13.0s)


  [ 2/18] Q2_2022.parquet    19,424,334 rows bucketed ( 14.4s)


  [ 3/18] Q3_2022.parquet    20,591,630 rows bucketed ( 11.8s)


...

  bucket  45/48  drives 378,624-387,229   11,821,472 rows  ( 12.5s)


  bucket  46/48  drives 387,229-395,834   13,563,211 rows  ( 13.6s)


  bucket  47/48  drives 395,834-404,439    9,879,549 rows  (  9.6s)


  bucket  48/48  drives 404,439-413,045    3,203,012 rows  (  3.8s)



ArchiveSeriesStore: 413,045 drives, 459,048,144 dense drive-days, 46 channels (84 GB memory-mapped), 18,564 drives with a failure event


In [6]:
splits = ap.grouped_drive_split(store, cfg)
checks = ap.verify_no_label_leakage(store, splits["train"], n_probe=20_000, seed=SEED)

LABEL-LEAKAGE VERIFICATION
  1. names ........... 46 channels, none named after `failure`
  2. storage ......... values -> values.f32, labels -> failure.u8 (separate files, no shared memory)
  3. independence .... labels overwritten with noise; X re-gathered bit-identical over 4,096 windows  PASS
  4. separability .... 20,000 probe windows, 0.395% positive
     strongest single channel: smart_197_raw at AUC 0.6961 (a proxy would sit at ~1.00)

  the model input is (batch, 30, 46) and contains SMART levels, their deltas and the observed mask -- nothing else


In [7]:
# The strongest channels, ranked against the window label. Useful signal sits well short
# of the 0.95 line; a proxy would be pinned against 1.00.
top = checks["channel_scores"].head(12).reset_index(drop=True)
top.index += 1
print("strongest single channels vs the 30-day window label "
      f"({checks['n_probe_windows']:,} probe windows):\n")
print(top.to_string())
print(f"\nceiling for a non-leaked channel: 0.95     observed maximum: "
      f"{checks['max_channel_auc']:.4f}")
print(f"model input shape: (batch, {cfg.window_days}, {checks['n_channels']}) "
      f"-- SMART levels, 1- and 7-day deltas, observed mask. No label channel.")

strongest single channels vs the 30-day window label (20,000 probe windows):

             channel  auc_vs_window_label
1      smart_197_raw             0.696141
2        smart_5_raw             0.682718
3      smart_198_raw             0.650253
4        smart_1_raw             0.631066
5        smart_7_raw             0.624499
6     smart_5_raw_d7             0.618820
7      smart_193_raw             0.603389
8        smart_3_raw             0.599879
9     smart_7_raw_d1             0.596560
10    smart_9_raw_d1             0.592520
11    smart_7_raw_d7             0.592028
12  smart_192_raw_d7             0.579903

ceiling for a non-leaked channel: 0.95     observed maximum: 0.6961
model input shape: (batch, 30, 46) -- SMART levels, 1- and 7-day deltas, observed mask. No label channel.


---
## 3 — Covering every drive's whole timeline with 30-day windows

Two different questions, two different window sets, both reported below.

**Tiling** lays non-overlapping 30-day blocks end to end along each drive's record:
$\lceil \text{record days} / 30 \rceil$ windows, so **every drive-day lands in exactly
one window** — nothing counted twice, nothing skipped. This is "as many batches as it
takes to fill the whole timeline", and it is what the decision tree in §5 is fitted on.
Where a record is not a multiple of 30, the last block is pulled back to end on the
drive's final day, so it overlaps its predecessor rather than running short.

**Stride-1 enumeration** is the other extreme: every possible start offset, used for the
sequence model's validation and test passes.

Both counts are printed, per split and per drive.

In [8]:
tiles = {name: ap.TilingWindows(store, splits[name], batch_size=cfg.batch_size,
                                split_name=name)
         for name in ("train", "val", "test")}

W = cfg.window_days
rows = []
for name in ("train", "val", "test"):
    idx, t = splits[name], tiles[name]
    rows.append({
        "split": name,
        "drives": len(idx),
        "drives with a failure": int(store.drive_has_failure[idx].sum()),
        "record days (dense)": int(store.lengths[idx].sum()),
        f"tiling {W}-day windows": t.n_windows,
        "tiling batches": len(t),
        "every possible window": int((store.lengths[idx] - W + 1).sum()),
    })
coverage = pd.DataFrame(rows).set_index("split")
print(coverage.to_string(formatters={c: "{:,}".format for c in coverage.columns
                                     if coverage[c].dtype.kind in "iu"}))

total_tiles = sum(t.n_windows for t in tiles.values())
total_batches = sum(len(t) for t in tiles.values())
print(f"\nTIMELINE COVERAGE")
print(f"  {store.n_drives:,} drives, {store.n_rows:,} dense drive-days")
print(f"  {total_tiles:,} non-overlapping {W}-day windows cover the entire timeline")
print(f"  at batch size {cfg.batch_size:,} that is {total_batches:,} batches in total")
for name in ("train", "val", "test"):
    print("  " + tiles[name].label_report())

       drives drives with a failure record days (dense) tiling 30-day windows tiling batches every possible window
split                                                                                                             
train 280,870                12,623         312,175,229            10,519,256          2,569           304,029,999
val    49,566                 2,228          55,215,887             1,860,545            455            53,778,473
test   82,609                 3,713          91,657,028             3,088,649            755            89,261,367

TIMELINE COVERAGE
  413,045 drives, 459,048,144 dense drive-days
  15,468,450 non-overlapping 30-day windows cover the entire timeline
  at batch size 4,096 that is 3,779 batches in total
  train | tiling W=30  |   10,519,256 windows | 280,870 drives | positive rate  0.2401% (25,254 positive windows) | 2,569 batches
  val   | tiling W=30  |    1,860,545 windows |  49,566 drives | positive rate  0.2391% (4,449 positive wi

In [9]:
# Per-drive accounting: how many windows each drive needs to have its whole record
# covered, and how much the final window had to overlap to stay a full 30 days.
cov = tiles["test"].coverage_report()
print(f"per-drive window counts, first 20 test drives "
      f"(batch size {cfg.batch_size:,}):\n")
print(cov.head(20).to_string(index=False))

n = cov["windows_to_cover_timeline"]
print(f"\nacross all {len(cov):,} test drives:")
print(f"  windows per drive .... min {n.min()}, median {int(n.median())}, "
      f"mean {n.mean():.1f}, max {n.max()}")
print(f"  total windows ........ {int(n.sum()):,}")
print(f"  total batches ........ {len(tiles['test']):,} "
      f"of up to {cfg.batch_size:,} windows each")
print(f"  days covered ......... {int(cov.days_covered.sum()):,} "
      f"(record holds {int(cov.record_days.sum()):,}; the difference is the "
      f"overlap pulled back onto the last window of each drive)")

per-drive window counts, first 20 test drives (batch size 4,096):

 drive_index    serial_number  record_days  windows_to_cover_timeline  days_covered  overlap_on_last_window  drive_has_failure
           3 02308a3415be0010         1642                         55          1650                       8              False
           7 0559e3d009d20010          993                         34          1020                      27              False
           8 0564f6f3fab90010         1642                         55          1650                       8              False
           9 059bba587f390010          993                         34          1020                      27              False
          15 078ea1af2c3f0010         1642                         55          1650                       8              False
          26 0e4aa019c4690010         1642                         55          1650                       8              False
          29     1030A004F9RG          252  

In [10]:
bundle = ap.build_archive_bundle(cfg, store=store, val_stride=1, verbose=True)


grouped split by drive id -- no drive appears in two splits:
  train  280,870 drives  (12,623 carry a failure, 4.49%)
  val     49,566 drives  (2,228 carry a failure, 4.50%)
  test    82,609 drives  (3,713 carry a failure, 4.49%)


  scaler fit on 40,000 train drives; rewriting 84 GB in place


    scaled 4,000,000/459,048,144 rows


    scaled 84,000,000/459,048,144 rows


    scaled 164,000,000/459,048,144 rows


    scaled 244,000,000/459,048,144 rows


    scaled 324,000,000/459,048,144 rows


    scaled 404,000,000/459,048,144 rows



Window datasets (window_days=30, horizon_days=30, stride_days=1):
  train | random spd 4   |    1,123,480 windows | 280,870 drives | positive rate  1.1078% (measured, 12,623 failing drives)
  val   | enumerate stride 1  |   53,778,473 windows |  49,566 drives | positive rate  0.1277% (68,656 positive windows)
  test  | enumerate stride 1  |   89,261,367 windows |  82,609 drives | positive rate  0.1287% (114,854 positive windows)

Raw pos_weight from the training positive rate: 89.3


---
## 4 — Training, then a verdict for every 30-day window

A 1D-CNN over the sequence: dilated temporal blocks so the receptive field spans the
full 30 days, then global pooling to one logit per window.
`(batch, 30, channels) → (batch, 1)`.

**One imbalance correction, not two.** The sampler forces 25% of the windows drawn from
failing drives to be positive ones, and the loss is plain BCE. Correcting twice — a
balanced sampler *and* a large `pos_weight` — is the mistake documented in
`TOSHIBA_PIPELINE.md`.

**Early stopping on validation PR-AUC.** At a base rate near 0.1%, accuracy is
meaningless: predicting "healthy" every time scores over 99.8%.

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tcfg = bw.TrainConfig(
    arch="cnn", hidden_dim=64, dropout=0.2,
    batch_size=cfg.batch_size, lr=1e-3, max_epochs=6, patience=2,
    loss="bce", sampler="none",
    # num_workers=0 on purpose: a batch is ~23 MB and worker IPC costs several times
    # what the gather itself does -- 8 workers measured 7x slower than none.
    num_workers=0, seed=SEED,
)

loaders = ap.build_loaders(bundle, tcfg, device)
model = bw.build_model(bundle.num_features, cfg.window_days, tcfg).to(device)
with torch.no_grad():
    probe = torch.zeros(2, cfg.window_days, bundle.num_features, device=device)
    print(f"input (2, {cfg.window_days}, {bundle.num_features}) -> "
          f"{tuple(model(probe).shape)}  (one logit per window)")
print(f"{bw.count_parameters(model):,} trainable parameters on {device}")

input (2, 30, 46) -> (2, 1)  (one logit per window)
136,769 trainable parameters on cuda


In [12]:
model, history, best_val = ap.train_archive_model(model, loaders, bundle, tcfg, device)
print(f"\nBest validation PR-AUC {best_val:.4f} at epoch "
      f"{1 + int(np.nanargmax(history['val_pr_auc']))}")
bw.plot_training_history(history, save_path=f"{FIG_DIR}/2022_step3_training_history.png")

Loss: BCEWithLogitsLoss(pos_weight=1.0)  [requested]


TRAINING -- CNN, 136,769 parameters, up to 6 epochs, early stop after 2 without a val PR-AUC gain
  275 training batches/epoch, 13,130 validation batches/epoch
epoch | train loss |  val loss | val PR-AUC | val ROC-AUC | val F1@.5 |       lr |     sec
------+------------+-----------+------------+-------------+-----------+----------+--------


      epoch 1 training  20.0%  loss 0.10554


      epoch 1 training  40.0%  loss 0.07801


      epoch 1 training  60.0%  loss 0.06795


...

    6 |    0.04624 |   0.01826 |     0.1513 |      0.8849 |    0.2301 | 1.00e-03 |   499.5  <- best



Best validation PR-AUC 0.1513 at epoch 6


  saved results/figures/2022_step3_training_history.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\backblaze_window_pipeline.py:2301: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1650x462 with 3 Axes>

In [13]:
results = ap.evaluate_archive_final(model, loaders, bundle, tcfg, device,
                                    pk_ks=(10, 25, 50, 100, 250, 1000))
results["history"] = history
ap.plot_evaluation(results, ks=(10, 25, 50, 100, 250, 1000),
                   save_path=f"{FIG_DIR}/2022_step3_evaluation.png")

FINAL VALIDATION PASS -- every window at stride 1


    val: 1/13,130 batches (  0.0%)  17,345 windows/s


    val: 657/13,130 batches (  5.0%)  158,012 windows/s


    val: 1,313/13,130 batches ( 10.0%)  159,472 windows/s


    val: 1,969/13,130 batches ( 15.0%)  160,307 windows/s


...

SUMMARY
  val   @tuned           PR-AUC 0.1513 | ROC-AUC 0.8849 | P 0.2280 | R 0.2749 | F1 0.2493 | thr 0.5826 | TP 18871 FP 63886 FN 49785 TN 53645931
  test  @0.50            PR-AUC 0.1441 | ROC-AUC 0.8859 | P 0.1682 | R 0.3253 | F1 0.2217 | thr 0.5000 | TP 37357 FP 184772 FN 77497 TN 88961741
  test  @tuned           PR-AUC 0.1441 | ROC-AUC 0.8859 | P 0.2205 | R 0.2589 | F1 0.2382 | thr 0.5826 | TP 29739 FP 105136 FN 85115 TN 89041377


  curves drawn from 4,000,000 of 89,261,367 test windows (uniform subsample; all printed metrics use the full pass)


  saved results/figures/2022_step3_evaluation.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\backblaze_window_pipeline.py:2376: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 2090x462 with 4 Axes>

### The test phase, one verdict per window

The threshold is the only quantity carried out of validation, and it was fixed before the
test split was touched. Below, every tiling window of every test drive gets a single
call — **did this drive fail within 30 days of this window ending, yes or no** — scored
against the truth.

This is the tiling window set, so each test drive-day is judged exactly once.

In [14]:
thr = results["best_threshold"]
pred = ap.predict_windows(model, tiles["test"], device, threshold=thr, tag="test")

tp = int(((pred.predicted_fail == 1) & (pred.actual_fail == 1)).sum())
fp = int(((pred.predicted_fail == 1) & (pred.actual_fail == 0)).sum())
fn = int(((pred.predicted_fail == 0) & (pred.actual_fail == 1)).sum())
tn = int(((pred.predicted_fail == 0) & (pred.actual_fail == 0)).sum())

print(f"\nPER-WINDOW VERDICTS -- {len(pred):,} tiling windows over "
      f"{pred.drive_index.nunique():,} test drives, threshold {thr:.4f}\n")
print(f"  predicted FAIL ....... {int(pred.predicted_fail.sum()):,}")
print(f"  actually  FAIL ....... {int(pred.actual_fail.sum()):,}")
print(f"  correct .............. {tp + tn:,} of {len(pred):,} "
      f"({100*(tp+tn)/len(pred):.4f}% accuracy)")
print(f"  TP {tp:,}   FP {fp:,}   FN {fn:,}   TN {tn:,}")
print(f"  precision {tp/max(tp+fp,1):.4f}   recall {tp/max(tp+fn,1):.4f}")
print(f"\n  note: 'accuracy' here is dominated by the {100*(1-pred.actual_fail.mean()):.3f}% "
      f"of windows that are negative -- always saying 'no fail' would score "
      f"{100*(1-pred.actual_fail.mean()):.4f}%.")

    test: 1/755 batches (  0.1%, 0s)


    test: 76/755 batches ( 10.1%, 2s)


    test: 151/755 batches ( 20.0%, 3s)


    test: 226/755 batches ( 29.9%, 5s)


    test: 301/755 batches ( 39.9%, 7s)


...

    test: 526/755 batches ( 69.7%, 11s)


    test: 601/755 batches ( 79.6%, 13s)


    test: 676/755 batches ( 89.5%, 15s)


    test: 751/755 batches ( 99.5%, 16s)



PER-WINDOW VERDICTS -- 3,088,649 tiling windows over 82,609 test drives, threshold 0.5826

  predicted FAIL ....... 6,461
  actually  FAIL ....... 7,424
  correct .............. 3,080,096 of 3,088,649 (99.7231% accuracy)
  TP 2,666   FP 3,795   FN 4,758   TN 3,077,430
  precision 0.4126   recall 0.3591

  note: 'accuracy' here is dominated by the 99.760% of windows that are negative -- always saying 'no fail' would score 99.7596%.


In [15]:
# Every window of a few drives that really failed: the verdict as the run produced it.
failed_drives = (pred[pred.actual_fail == 1].drive_index.drop_duplicates().head(3))
for gidx in failed_drives:
    sub = pred[pred.drive_index == gidx].sort_values("window_start")
    print(f"\ndrive {sub.serial_number.iloc[0]}  --  {len(sub)} windows "
          f"covering {sub.window_start.min():%Y-%m-%d} .. {sub.window_end.max():%Y-%m-%d}")
    show = sub[["window_in_drive", "window_start", "window_end", "risk",
                "predicted_fail", "actual_fail"]].copy()
    show["verdict"] = np.where(show.predicted_fail == 1, "FAIL", "no fail")
    show["truth"] = np.where(show.actual_fail == 1, "FAIL", "no fail")
    print(show.tail(12)[["window_in_drive", "window_start", "window_end",
                         "risk", "verdict", "truth"]].to_string(index=False))


drive 1040A00VF97G  --  36 windows covering 2022-01-01 .. 2024-12-01
 window_in_drive window_start window_end     risk verdict   truth
              24   2023-12-22 2024-01-20 0.005965 no fail no fail
              25   2024-01-21 2024-02-19 0.006689 no fail no fail
              26   2024-02-20 2024-03-20 0.005041 no fail no fail
              27   2024-03-21 2024-04-19 0.006211 no fail no fail
              28   2024-04-20 2024-05-19 0.004789 no fail no fail
              29   2024-05-20 2024-06-18 0.006140 no fail no fail
              30   2024-06-19 2024-07-18 0.004966 no fail no fail
              31   2024-07-19 2024-08-17 0.005465 no fail no fail
              32   2024-08-18 2024-09-16 0.004593 no fail no fail
              33   2024-09-17 2024-10-16 0.004752 no fail no fail
              34   2024-10-17 2024-11-15 0.007960 no fail    FAIL
              35   2024-11-02 2024-12-01 0.006002 no fail    FAIL

drive 1040A0C9F97G  --  8 windows covering 2022-01-01 .. 2022-08-01
 wi

In [16]:
fig, picks = ap.plot_drive_histograms(
    pred, n_drives=24, threshold=thr, seed=SEED,
    save_path=f"{FIG_DIR}/2022_step4_drive_histograms.png")
print(f"\n{len(picks)} drives plotted, one histogram each: the distribution of that "
      f"drive's 30-day windows over predicted risk.")

  saved results/figures/2022_step4_drive_histograms.png

24 drives plotted, one histogram each: the distribution of that drive's 30-day windows over predicted risk.


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\archive_window_pipeline.py:1940: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5 — A decision tree grown until it is perfect on its training set

Fitted on the **tiling** windows, so the tree sees every training drive-day exactly once.
Each window is reduced to a row of summary features — per channel, its last day, its
30-day mean, and its first-to-last change — because a tree cannot take a `(30, 46)`
tensor.

The depth ladder below is the point of the section. A tree with no depth limit and
`min_samples_leaf=1` will reach **1.000 on the data it was fitted to** by carving a leaf
per sample; that is what "as big as needed for 100%" produces, and the test column beside
it is what it costs. Both are printed at every depth so the gap is visible rather than
asserted.

Two things to read carefully:

* **Accuracy is the wrong lens at this base rate.** Predicting "never fails" scores about
  99.9%. Balanced accuracy — the mean of per-class recall — is reported alongside, and it
  is the column that moves.
* **100% may be unreachable and that is informative.** If two windows have identical
  summary features but different labels, no tree can separate them. The cell reports how
  many such conflicts exist, which is the ceiling on training accuracy.

In [17]:
t0 = time.time()
Xtr, ytr, feat_cols = ap.window_table_features(
    store, tiles["train"], aggregates=("last", "mean", "delta"))
Xte, yte, _ = ap.window_table_features(
    store, tiles["test"], aggregates=("last", "mean", "delta"))
print(f"\nbuilt in {time.time()-t0:.0f}s   train {Xtr.shape}   test {Xte.shape}")
print(f"{len(feat_cols)} features = {len(store.feature_names)} channels x "
      f"3 aggregates (last, mean, delta)")

    features 500,000/10,519,256 (  4.8%, 12s)


    features 2,500,000/10,519,256 ( 23.8%, 53s)


    features 4,500,000/10,519,256 ( 42.8%, 101s)


    features 6,500,000/10,519,256 ( 61.8%, 171s)


    features 8,500,000/10,519,256 ( 80.8%, 269s)


    features 10,500,000/10,519,256 ( 99.8%, 411s)


  feature table 10,519,256 x 138 float32 (5.8 GB), 25,254 positive (0.2401%)


    features 500,000/3,088,649 ( 16.2%, 19s)


    features 2,500,000/3,088,649 ( 80.9%, 27s)


  feature table 3,088,649 x 138 float32 (1.7 GB), 7,424 positive (0.2404%)

built in 447s   train (10519256, 138)   test (3088649, 138)
138 features = 46 channels x 3 aggregates (last, mean, delta)


In [18]:
# The exact ceiling: rows identical in every feature but disagreeing on the label can
# never be separated by any tree, however deep it is allowed to grow.
ceiling = ap.count_label_conflicts(Xtr, ytr)

  39 groups of identical feature rows carry both labels (13,981 rows, 42s)
  46 rows can never be classified correctly by any tree
  => ceiling on training accuracy: 0.99999563


In [19]:
tree, ladder = ap.grow_tree_to_full_accuracy(
    Xtr, ytr, Xte, yte, depths=[4, 8, 16, 32], seed=SEED)
print()
print(ladder.to_string(index=False))

  depth    4 -> actual   4,        16 leaves | train acc 0.997777 (bal 0.5552) | test acc 0.997766 (bal 0.5538) | 193s


  depth    8 -> actual   8,       232 leaves | train acc 0.997925 (bal 0.6065) | test acc 0.997805 (bal 0.5936) | 383s


  depth   16 -> actual  16,     2,997 leaves | train acc 0.998491 (bal 0.6917) | test acc 0.997671 (bal 0.6057) | 749s


  depth   32 -> actual  32,     8,186 leaves | train acc 0.999008 (bal 0.7941) | test acc 0.997295 (bal 0.6218) | 1486s


  depth none -> actual 123,    28,881 leaves | train acc 0.999996 (bal 0.9991) | test acc 0.996040 (bal 0.6320) | 4816s



max_depth  actual_depth  leaves  train_accuracy  test_accuracy  train_balanced_acc  test_balanced_acc  fit_seconds
        4             4      16        0.997777       0.997766            0.555234           0.553834   193.245393
        8             8     232        0.997925       0.997805            0.606543           0.593628   382.889183
       16            16    2997        0.998491       0.997671            0.691718           0.605655   749.305925
       32            32    8186        0.999008       0.997295            0.794130           0.621793  1486.263070
     none           123   28881        0.999996       0.996040            0.999129           0.632049  4816.425097


In [20]:
final = ladder.iloc[-1]
print(f"FINAL TREE -- depth {final.actual_depth}, {final.leaves:,} leaves, "
      f"no depth limit, min_samples_leaf=1")
print(f"  training accuracy .... {final.train_accuracy:.8f}")
print(f"  test accuracy ........ {final.test_accuracy:.8f}")
print(f"  training balanced .... {final.train_balanced_acc:.4f}")
print(f"  test balanced ........ {final.test_balanced_acc:.4f}")
print()
if final.train_accuracy >= 1.0:
    print("  reached 100.000000% on the training set: one leaf per distinguishable row.")
else:
    missed = int(round((1 - final.train_accuracy) * len(ytr)))
    print(f"  it lands {missed:,} rows short of 100%, and those are the "
          f"{ceiling['unreachable_rows']:,} rows that are")
    print(f"  IDENTICAL across all {Xtr.shape[1]} features to a row carrying the "
          f"opposite label. No tree can")
    print(f"  split what it cannot tell apart, so "
          f"{100*ceiling['max_train_accuracy']:.6f}% is the ceiling -- the tree is")
    print(f"  already at it, not falling short of it.")
print(f"\n  the gap between the two ACCURACY columns is the tree memorising "
      f"{final.leaves:,} leaves of training rows;")
print(f"  the gap between the two BALANCED columns "
      f"({final.train_balanced_acc:.3f} vs {final.test_balanced_acc:.3f}) is what that "
      f"memorisation is")
print(f"  actually worth on a disk it has never seen.\n")

imp = pd.DataFrame({"feature": feat_cols, "importance": tree.feature_importances_})
imp = imp[imp.importance > 0].sort_values("importance", ascending=False).head(15)
imp.index = range(1, len(imp) + 1)
print("what the tree splits on most:")
print(imp.to_string())

FINAL TREE -- depth 123, 28,881 leaves, no depth limit, min_samples_leaf=1
  training accuracy .... 0.99999563
  test accuracy ........ 0.99604034
  training balanced .... 0.9991
  test balanced ........ 0.6320

  it lands 46 rows short of 100%, and those are the 46 rows that are
  IDENTICAL across all 138 features to a row carrying the opposite label. No tree can
  split what it cannot tell apart, so 99.999563% is the ceiling -- the tree is
  already at it, not falling short of it.

  the gap between the two ACCURACY columns is the tree memorising 28,881 leaves of training rows;
  the gap between the two BALANCED columns (0.999 vs 0.632) is what that memorisation is
  actually worth on a disk it has never seen.

what the tree splits on most:
                    feature  importance
1    smart_197_raw_d7__last    0.071871
2      smart_5_raw_d1__last    0.037265
3       smart_194_raw__mean    0.025644
4     smart_9_raw_d7__delta    0.022574
5   smart_193_raw_d7__delta    0.019868
6     s

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
x = np.arange(len(ladder))
ax.plot(x, ladder.train_accuracy, "o-", color="#3b6ea5", label="train accuracy")
ax.plot(x, ladder.test_accuracy, "o-", color="#c1442e", label="test accuracy")
ax.set_xticks(x); ax.set_xticklabels(ladder.max_depth.astype(str))
ax.set_xlabel("max_depth"); ax.set_ylabel("accuracy")
ax.set_title("Accuracy is the wrong lens here\n(both columns pinned near 1.0 by the base rate)")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(x, ladder.train_balanced_acc, "o-", color="#3b6ea5", label="train balanced acc")
ax.plot(x, ladder.test_balanced_acc, "o-", color="#c1442e", label="test balanced acc")
ax.set_xticks(x); ax.set_xticklabels(ladder.max_depth.astype(str))
ax.set_xlabel("max_depth"); ax.set_ylabel("balanced accuracy")
ax.set_title("Balanced accuracy: the gap that opens is the overfitting")
ax.legend(fontsize=8)
for a in axes:
    a.grid(alpha=0.25, linewidth=0.6); a.set_axisbelow(True)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/2022_step5_tree_depth.png", dpi=130, bbox_inches="tight")
print(f"  saved {FIG_DIR}/2022_step5_tree_depth.png")
plt.show()

  saved results/figures/2022_step5_tree_depth.png


C:\Users\tempuser2\AppData\Local\Temp\7\ipykernel_114592\1361581098.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
os.makedirs("models", exist_ok=True); os.makedirs("results", exist_ok=True)
ap.save_artifacts(model, bundle, tcfg, results, prefix="models/backblaze_2022_w30")

pd.DataFrame(history).to_csv("results/w2022_training_history.csv", index=False)
coverage.to_csv("results/w2022_window_coverage.csv")
cov.to_csv("results/w2022_per_drive_windows.csv", index=False)
ladder.to_csv("results/w2022_tree_depth_ladder.csv", index=False)
results["top_drives"].to_csv("results/w2022_top_drives.csv", index=False)
checks["channel_scores"].to_csv("results/w2022_channel_label_auc.csv", index=False)
pred.head(2_000_000).to_csv("results/w2022_window_verdicts.csv", index=False)
pd.concat([results["precision_at_k_windows"].assign(ranked="windows"),
           results["precision_at_k_drives"].assign(ranked="drives")],
          ignore_index=True).to_csv("results/w2022_precision_at_k.csv", index=False)

with open("results/w2022_summary.json", "w", encoding="utf-8") as fh:
    json.dump({
        "since_year": SINCE_YEAR,
        "shards": [os.path.basename(p)[:-8] for p in shards],
        "features": list(features),
        "blocklist": report.blocklist,
        "n_drives": int(store.n_drives),
        "n_drive_days": int(store.n_rows),
        "tiling_windows_total": int(total_tiles),
        "tiling_batches_total": int(total_batches),
        "leakage_checks_passed": bool(checks["x_independent_of_label"]),
        "max_channel_auc_vs_label": float(checks["max_channel_auc"]),
        "best_threshold": results["best_threshold"],
        "test_at_tuned": results["test_at_tuned"],
        "tree_ceiling": {k: float(v) for k, v in ceiling.items()},
        "tree_final": {k: (float(v) if isinstance(v, (int, float, np.floating)) else str(v))
                       for k, v in ladder.iloc[-1].to_dict().items()},
    }, fh, indent=2)
print("wrote results/w2022_*.csv and results/w2022_summary.json")

Saved models/backblaze_2022_w30_cnn.pth and models/backblaze_2022_w30_preprocessing_config.json


wrote results/w2022_*.csv and results/w2022_summary.json


---
## What these numbers do and do not say

**The label never reached the model.** §2 is an executed experiment, not a promise: the
labels were replaced with noise and the window tensors came back bit-identical. Together
with the audit's blocklist, that is the strongest statement available short of a proof —
the model's ranking comes from SMART telemetry and nothing else.

**The ranking transfers; the base rate is already real.** This is the fleet at its own
prevalence, not a matched cohort, so the precision figures need no discounting before
they are read. Read the **lift** column in Precision@K rather than raw precision, and
prefer the drive-level table — an operator pulls *K disks*, not K windows.

**Tiling and stride-1 answer different questions.** The tiling set judges every
drive-day exactly once, which is what makes the per-window verdict table and the tree
honest. The stride-1 set scores every window that exists, which is the right basis for a
ranking metric. Neither is a sample of the other.

**The 100% tree is a memorisation result, not a model.** It reaches perfect training
accuracy by growing a leaf per training row, and the test column next to it is the honest
read. At a 0.1% base rate even the test accuracy is near-perfect while the model is
nearly useless — which is exactly why every other number in this notebook is PR-AUC,
Precision@K or balanced accuracy rather than accuracy.

**2022 onward is not the whole archive.** Drive models, capacities and the SMART
attributes actually populated all shift over time; a model fitted on the last few years
is answering a narrower and more current question than one fitted on all thirteen.